# Урок 32 — Threading & Concurrency: Як Python керує потоками

## Race Conditions, GIL, Locks, Deadlocks та ThreadPoolExecutor

**Модуль 4 · Network & Concurrent Systems · Автор: Viktor Nikoriak**

---

# Про що цей урок?

У попередньому уроці ми навчились робити HTTP запити:

- `requests.get()` блокує потік
- Поки один запит виконується — Python process завмирає
- Якщо треба 100 запитів — вони виконуються послідовно?

**Ні.** Є кращий спосіб.

Більшість Python-розробників у какийсь момент пишуть:

```python
import threading

t = threading.Thread(target=fetch_data, args=(url,))
t.start()
```

І це виглядає просто.

Але за цим ховаються:

- OS thread scheduler (не-детерміністичний)
- Global Interpreter Lock (GIL)
- Race conditions (тихе пошкодження даних)
- Deadlocks (програма зависає навічно)
- Context switches (OS перериває потік у будь-який момент)

---

# Головна мета уроку

> **Перестати бачити код як послідовний.** Навчитись ментально симулювати хаотичне, недетерміністичне виконання кількох потоків.

Ти навчишся:

- **Розуміти** що насправді відбувається коли Python виконує два потоки
- **Бачити** де виникають race conditions і чому
- **Фіксити** проблеми за допомогою Lock, RLock, Semaphore
- **Уникати** deadlocks через правильний lock ordering
- **Писати** production-grade concurrent код з ThreadPoolExecutor

---

# Де це використовується в реальних системах?

| Система | Як використовується threading |
| ------- | ----------------------------- |
| Web Servers (Gunicorn) | Пул потоків — кожен обробляє один HTTP запит |
| Data Pipelines | Паралельне завантаження файлів або API запитів |
| GUI Applications | Background задачі щоб UI не зависав |
| Celery Workers | Thread pool для виконання async tasks |
| FastAPI | ThreadPoolExecutor для sync endpoint функцій |

---

# Важлива передумова

Цей урок передбачає, що ти вже розумієш:

- що таке blocking I/O (Урок 30)
- що таке HTTP запит (Урок 31)
- базовий Python (функції, глобальні змінні, виключення)

# Learning Objectives

Після цього уроку ти зможеш:

## Conceptual Understanding

- Пояснити різницю між **Process** та **Thread** на рівні пам'яті
- Описати що таке **GIL** і чому він існує в CPython
- Пояснити чому `counter += 1` **не є** атомарною операцією
- Описати всі 4 умови **Coffman**, що призводять до Deadlock
- Пояснити різницю між **Stack** (локальні змінні) та **Heap** (спільна пам'ять)

## Debugging Skills

- Передбачити результат коду з race condition
- Визначити **де саме** потрібен Lock у коді
- Розпізнати pattern, що призводить до Deadlock
- Пояснити чому Heisenbug зникає при додаванні `print()`

## Production Skills

- Написати thread-safe код з `threading.Lock`
- Використовувати `ThreadPoolExecutor` для concurrent HTTP запитів
- Налаштувати `logging` замість `print()` для debugging
- Побудувати Producer/Consumer pipeline з `queue.Queue`

# Частина 3: Перший потік — Недетермінізм

## Level 1: Потоки можуть завершуватись у різному порядку

У звичайному Python-коді команди виконуються зверху вниз.

Але коли ми запускаємо кілька потоків, вони не обов’язково завершуються в тому порядку, в якому були створені.

Ми можемо створити потоки так:

```text
Worker 0
Worker 1
Worker 2
Worker 3
Worker 4
````

Але завершитись вони можуть так:

```text
Worker 3
Worker 1
Worker 4
Worker 0
Worker 2
```

Це нормально.

Кожен потік виконує роботу різну кількість часу:

```python
duration = random.uniform(0.1, 0.5)
time.sleep(duration)
```

Тому потік, який стартував першим, не завжди завершується першим.

Головна ідея:

> Старт потоків ≠ порядок завершення потоків.

Запусти код нижче кілька разів і подивись, як змінюється порядок завершення воркерів.





In [ ]:
import threading
import time
import random

def worker(worker_id):
    """Воркер, що симулює роботу різної тривалості."""
    # Різний час виконання — імітація реального IO
    duration = random.uniform(0.1, 0.5)
    print(f"[Воркер {worker_id}] Стартував, буде працювати {duration:.2f}с")
    time.sleep(duration)
    print(f"[Воркер {worker_id}] Завершив роботу")

# Запускаємо 5 потоків
threads = []
for i in range(5):
    t = threading.Thread(target=worker, args=(i,), name=f"Worker-{i}")
    threads.append(t)
    t.start()

# Чекаємо поки всі завершать роботу
for t in threads:
    t.join()

print("\n✅ Всі потоки завершили роботу")

## Що треба помітити?

`Worker 0` може стартувати першим, але завершитись останнім.

Це називається **недетермінізм**:
той самий код може давати різний порядок результатів при різних запусках.

Це важливо, бо далі ми побачимо:
якщо кілька потоків одночасно змінюють одну змінну, результат теж може бути непередбачуваним.

## Пояснення: Що щойно відбулось?

**Ключові спостереження:**

1. **Порядок завершення непередбачуваний** — кожен запуск дає інший результат
2. **`.start()`** — потік переходить у стан `Runnable`. OS вирішує КОЛИ дати йому CPU
3. **`.join()`** — main thread чекає поки цей потік перейде у стан `Dead`
4. **`time.sleep()`** — відпускає GIL, потік чекає без використання CPU

**Стани потоку:**

```
Thread(target=fn)  →  .start()  →  OS scheduling  →  .join()
     New          →  Runnable  →    Running        →   Dead
                                       ↕
                                   Waiting (під час IO)
```

**Debugging tip:** Якщо `.join()` не викликати — main thread може завершитись до child threads → корупція даних або незавершені операції.

**Типова помилка:**
```python
# ❌ Забув join() — потоки можуть не завершитись
for t in threads:
    t.start()
print("Готово")  # Виводиться ДО завершення потоків!

# ✅ Правильно
for t in threads:
    t.start()
for t in threads:
    t.join()  # Чекаємо кожен потік
```

# Частина 4: IO-bound — Як потоки "імітують" паралелізм

## Порівняння: Послідовно vs Конкурентно

Подивимось скільки часу займає 5 "мережевих запитів" послідовно і конкурентно.

In [ ]:
import threading
import time
from tqdm import tqdm

def fetch_data(worker_id, results):
    """Симуляція повільного мережевого запиту."""
    time.sleep(2)
    results[worker_id] = f'Дані від {worker_id}'

# ── Послідовне виконання ────────────────────────────────────────────────────
print('ПОСЛІДОВНЕ')
start = time.perf_counter()
results_sequential = {}
for i in tqdm(range(5), desc='Sequential', unit='req', colour='red'):
    fetch_data(f'seq-{i}', results_sequential)
elapsed_seq = time.perf_counter() - start
print(f'Час: {elapsed_seq:.2f}с\n')

# ── Конкурентне виконання ───────────────────────────────────────────────────
print('КОНКУРЕНТНЕ (threading)')
start = time.perf_counter()
results_concurrent = {}
threads = []
for i in range(5):
    t = threading.Thread(target=fetch_data, args=(f'thr-{i}', results_concurrent))
    threads.append(t)
    t.start()

# tqdm на join() — бачимо скільки потоків вже завершились
for t in tqdm(threads, desc='Joining threads', unit='thread', colour='green'):
    t.join()

elapsed_conc = time.perf_counter() - start
print(f'Час: {elapsed_conc:.2f}с')
print(f'\n🚀 Прискорення: {elapsed_seq / elapsed_conc:.1f}x')


## Пояснення: Чому threading прискорив IO?

**Покроковий Timeline конкурентного виконання:**

```
[T=0]   main: створює 5 потоків, викликає .start()
         → всі 5 потоків переходять у Runnable

[T=0.01] Thread 0: захоплює GIL, друкує "Починаю запит..."
[T=0.01] Thread 0: hits time.sleep(2) → ВІДПУСКАЄ GIL → стан: Waiting

[T=0.01] Thread 1: захоплює вільний GIL, друкує, sleep → Waiting
[T=0.01] Thread 2: захоплює GIL, друкує, sleep → Waiting
[T=0.01] Thread 3: захоплює GIL, друкує, sleep → Waiting
[T=0.01] Thread 4: захоплює GIL, друкує, sleep → Waiting

         ... 2 секунди тиші. Всі 5 потоків чекають IO паралельно ...

[T=2.01] Thread 0: OS будить → захоплює GIL → друкує "Отримав відповідь!"
[T=2.01] Thread 1: OS будить → захоплює GIL → друкує
         ...

Загальний час: ~2 секунди замість 10!
```

**Ключова інтуїція:** Потоки не виконуються в одну мікросекунду. Але вони **елегантно передають GIL** коли змушені чекати. Це і є concurrency — не parallelism, а ефективне використання часу очікування.

# Частина 5: Race Condition — Тихе Пошкодження Даних

## Level 2: Відтворення Race Condition

Зупинись. Перед тим як запустити — **передбач результат**.

### Prediction Quiz

Два потоки, кожен інкрементує лічильник 1,000,000 разів.
Що виведе `print(counter)` в кінці?

- **A)** Рівно 2,000,000
- **B)** Рівно 1,000,000  
- **C)** Випадкове число, майже завжди менше 2,000,000
- **D)** Падіння з `ConcurrentModificationError`

*Відповідь після коду.*

In [ ]:
import threading
import time

counter = 0

def increment():
    global counter

    for _ in range(1_000_000):
        temp = counter      # READ

        # ШТУЧНО даємо іншому потоку шанс
        time.sleep(0)

        temp = temp + 1     # MODIFY

        counter = temp      # WRITE

threads = [
    threading.Thread(target=increment),
    threading.Thread(target=increment)
]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(f"Очікували: 2,000,000")
print(f"Отримали:  {counter:,}")
print(f"Втрачено:  {2_000_000 - counter:,}")

## Відповідь: **C — Випадкове число менше 2,000,000**

## Чому новачки помиляються

Новачки думають `counter += 1` — **атомарна** операція. Це МІФ.

Реальність: `counter += 1` компілюється у **4 байткодові інструкції**:

```python
import dis
dis.dis("counter += 1")

#  LOAD_NAME    counter    ← 1. Читати значення з пам'яті
#  LOAD_CONST   1          ← 2. Завантажити константу 1
#  BINARY_OP    +          ← 3. Додати
#  STORE_NAME   counter    ← 4. Записати результат
```

OS може зробити **context switch між будь-якими двома** з цих інструкцій.

## Покрокова Колізія

```
Час | Thread 1 (T1)              | Thread 2 (T2)          | counter
----+-----------------------------+------------------------+--------
 1  | READS counter → бачить 10  |                        | 10
 2  | [OS PREEMPTION! T1 пауза]  | [T2 отримує CPU]       | 10
 3  | (спить з числом 10 у голові)| READS counter → бачить 10 | 10
 4  |                            | ADDS 1 → 11            | 10
 5  |                            | WRITES 11              | 11
 6  | [OS відновлює T1]          |                        | 11
 7  | ADDS 1 до СТАРОГО 10 → 11  |                        | 11
 8  | WRITES 11 ← ПЕРЕЗАПИСУЄ T2!|                        | 11 ❌

Результат: 2 increment-и, але counter підвищився тільки на 1!
```

## Банківська аналогія

Уяви banking app: баланс £2,000. Thread A (payroll) депозить £5,000. Thread B (оренда) знімає £1,000. Обидва читають £2,000. Thread B оновлює до £1,000. Thread A перезаписує до £7,000. **Платіж за оренду втрачено назавжди.**

**Debugging tip:** Race condition не відтворюється стабільно. Локально може "працювати", на prod під навантаженням — ламається. Завжди перевіряй спільні змінні.

# Частина 6: Lock — Виправляємо Race Condition

## Що таке Lock (Mutex)

**Lock** — булевий прапорець на рівні OS. Коли потік намагається `acquire()` заблокований Lock:
1. OS переводить потік у **Waiting state** (0% CPU)
2. Прибирає з черги виконання
3. Розбудить тільки коли власник Lock викличе `release()`

**Critical Section** — ділянка коду між `acquire()` та `release()`, де тільки **один потік** може бути одночасно.

In [ ]:
import threading

counter = 0
lock = threading.Lock()  # Один Lock на весь спільний ресурс

def safe_increment():
    global counter
    for _ in range(1_000_000):
        with lock:        # Автоматично acquire() та release()
            counter += 1  # Critical Section: тільки один потік тут

t1 = threading.Thread(target=safe_increment)
t2 = threading.Thread(target=safe_increment)

t1.start()
t2.start()

t1.join()
t2.join()

print(f"Очікували: 2,000,000")
print(f"Отримали:  {counter:,}")
print(f"✅ Race condition виправлено!")

## Пояснення: Як Lock виправляє проблему

```mermaid
sequenceDiagram
    participant T1 as 🧵 Thread 1
    participant LOCK as 🔐 Lock
    participant MEM as 💾 counter
    participant T2 as 🧵 Thread 2

    T1->>LOCK: acquire() ✅
    T2->>LOCK: acquire() ❌ ЗАБЛОКОВАНО
    Note over T2: Thread 2 спить (0% CPU)

    T1->>MEM: READ, ADD 1, WRITE
    T1->>LOCK: release() — будить T2

    T2->>LOCK: acquire() ✅
    T2->>MEM: READ (бачить оновлене!), ADD 1, WRITE ✅
    T2->>LOCK: release()
```

**Трейдоф:** Lock перетворює паралельний код на **послідовний** для critical section. Це bottleneck. Тому critical section має бути **мінімально можливою**.

## Практичний приклад: Банківський рахунок

In [ ]:
import threading
import time

bank_balance = 100
lock = threading.Lock()

def withdraw(amount, user):
    """Thread-safe зняття коштів."""
    global bank_balance

    with lock:  # Lock захищає весь read-check-write цикл
        print(f"[{user}] Перевіряю баланс: {bank_balance}£")

        if bank_balance >= amount:
            # Симуляція мережевої затримки під час transaction
            time.sleep(0.05)
            bank_balance -= amount
            print(f"[{user}] Знято {amount}£. Залишок: {bank_balance}£")
        else:
            print(f"[{user}] Недостатньо коштів!")

# Два користувачі намагаються зняти 80£ одночасно (баланс 100£)
t1 = threading.Thread(target=withdraw, args=(80, "User A"))
t2 = threading.Thread(target=withdraw, args=(80, "User B"))

t1.start()
t2.start()

t1.join()
t2.join()

print(f"\nФінальний баланс: {bank_balance}£")
print("(Правильно: тільки один успішно зняв 80£)")

## Пояснення: Де саме потрібен Lock?

**Read-Check-Write pattern** — найнебезпечніший паттерн:

```python
# ❌ Небезпечно — три кроки без захисту
current = bank_balance      # 1. READ
if current >= amount:       # 2. CHECK
    bank_balance -= amount  # 3. WRITE

# ✅ Безпечно — весь цикл в одному Lock
with lock:
    if bank_balance >= amount:
        bank_balance -= amount
```

**Типова помилка:** Lock тільки навколо WRITE, але не READ+CHECK:

```python
# ❌ НЕПРАВИЛЬНО — читаємо без захисту!
if bank_balance >= amount:  # Thread 2 теж читає тут
    with lock:
        bank_balance -= amount
```

**Debugging tip:** Завжди думай про **весь атомарний блок** операцій, а не тільки про write. Read-Check-Write має бути atomic разом.

# Частина 7: Deadlock — Програма, що Зависає Назавжди

## Level 3: Системне Мислення

### Prediction Quiz

Що виведе цей код?

- **A)** Обидва таски виконаються успішно
- **B)** Програма впаде з `DeadlockError`
- **C)** Task 1 завершиться, Task 2 — ні
- **D)** Програма зависне назавжди і ніколи не завершиться

*Відповідь після коду.*

In [ ]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def task_1():
    print("[Task 1] Захоплюю lock_a...")
    with lock_a:
        print("[Task 1] lock_a захоплено. Чекаю 0.1с...")
        time.sleep(0.1)
        print("[Task 1] Намагаюсь захопити lock_b...")
        with lock_b:  # Task 2 вже тримає lock_b!
            print("[Task 1] Завершив!")  # Сюди ніколи не дійдемо

def task_2():
    print("[Task 2] Захоплюю lock_b...")
    with lock_b:
        print("[Task 2] lock_b захоплено. Чекаю 0.1с...")
        time.sleep(0.1)
        print("[Task 2] Намагаюсь захопити lock_a...")
        with lock_a:  # Task 1 вже тримає lock_a!
            print("[Task 2] Завершив!")  # Сюди ніколи не дійдемо

t1 = threading.Thread(target=task_1)
t2 = threading.Thread(target=task_2)

t1.start()
t2.start()

# УВАГА: Ці join() ніколи не завершяться (deadlock)!
# Запустимо з timeout щоб не зависнути в notebook
t1.join(timeout=2)
t2.join(timeout=2)

if t1.is_alive() or t2.is_alive():
    print("\n💀 DEADLOCK виявлено! Потоки все ще висять.")
    print(f"   Task 1 живий: {t1.is_alive()}")
    print(f"   Task 2 живий: {t2.is_alive()}")

## Відповідь: **D — Програма зависає назавжди**

## Покроковий Timeline Deadlock

```
[T=0.00] Task 1: захоплює lock_a ✅
[T=0.00] Task 2: захоплює lock_b ✅

[T=0.10] Task 1: прокидається, намагається lock_b → ❌ BLOCKED (Task 2 тримає!)
          → Task 1 переходить у Waiting state, 0% CPU

[T=0.10] Task 2: прокидається, намагається lock_a → ❌ BLOCKED (Task 1 тримає!)
          → Task 2 переходить у Waiting state, 0% CPU

          💀 DEADLOCK:
          Task 1 чекає Task 2 (lock_b)
          Task 2 чекає Task 1 (lock_a)
          Жоден ніколи не прокинеться.
          Програма зависла назавжди.
```

## Чому немає `DeadlockError`?

Python **не має** вбудованого детектора deadlock. Потоки просто **сплять вічно**. Програма споживає 0% CPU але ніколи не завершується. Це найнебезпечніший вид багу — не видно, не кидає виключень.

## Рішення: Lock Ordering

In [ ]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def acquire_in_order(*locks):
    """Завжди захоплює locks у стабільному порядку (за id об'єкта)."""
    # Сортуємо по id — гарантований детермінований порядок
    return sorted(locks, key=id)

def safe_task_1():
    # Обидва таски захоплюють locks в ОДНОМУ порядку
    first, second = acquire_in_order(lock_a, lock_b)
    with first:
        time.sleep(0.1)
        with second:
            print("[Safe Task 1] Завершив! ✅")

def safe_task_2():
    # Той самий порядок — незалежно від того які locks передані
    first, second = acquire_in_order(lock_b, lock_a)  # порядок аргументів не важливий
    with first:
        time.sleep(0.1)
        with second:
            print("[Safe Task 2] Завершив! ✅")

t1 = threading.Thread(target=safe_task_1)
t2 = threading.Thread(target=safe_task_2)

t1.start()
t2.start()

t1.join()
t2.join()

print("\n✅ Deadlock виправлено через lock ordering!")

## Пояснення: Чому Lock Ordering вирішує проблему

**Правило:** Всі потоки завжди захоплюють кілька locks у **одному порядку**.

Якщо обидва потоки хочуть `lock_a` і `lock_b`, і обидва намагаються спочатку `lock_a` — тільки **один** з них успішно захопить його. Другий чекатиме, поки перший не завершить і не відпустить обидва.

**Правило "Дайнінг Філософів":** Розсади філософів за столом і скажи одному з них брати СПОЧАТКУ праву виделку. Тепер не може бути circular wait.

**Debugging tip:** Heisenbug deadlock — deadlock, що зникає при малому навантаженні. При 2 потоках може не проявитись, при 50 — зависне. Завжди перевіряй порядок lock acquisition в code review.

# Частина 8: Identity Crisis — Stack vs Heap

## Exercise 3: Передбач — чи потік надрукує свій власний ID?

### Prediction Quiz

Чи потік надрукує `thread_id`, що **не** відповідає його власному?

- **A)** Ні — `thread_id` переданий локально, він завжди свій
- **B)** Так — потоки надрукують чужий ID
- **C)** Ні — присвоєння словника в Python атомарне

In [ ]:
import threading
import time

# Heap — спільна пам'ять для ВСІХ потоків
shared_state = {}

def update_state(thread_id):
    # Записує свій ID у спільний словник
    shared_state['last_active'] = thread_id

    # Симуляція роботи — за цей час інші потоки перезаписують словник
    time.sleep(0.01)

    # Читає 'last_active' — але це вже значення ІНШОГО потоку!
    print(
        f"Я потік {thread_id}, "
        f"last_active = {shared_state['last_active']}  "
        f"{'← ✅ збіг' if shared_state['last_active'] == thread_id else '← ❌ ЧУЖИЙ!'}"
    )

threads = [threading.Thread(target=update_state, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

## Відповідь: **B — Потоки надрукують чужий ID**

## Чому виникає ця плутанина?

Новачки плутають **локальний стек** (приватний) зі **спільною купою** (Heap).

```
thread_id = 0   ← Stack (приватна копія, безпечна!)
shared_state    ← Heap (одна копія для ВСІХ потоків!)
```

**Покрокова колізія:**
```
[T=1] Потік 0: shared_state['last_active'] = 0, йде спати
[T=2] Потік 1: shared_state['last_active'] = 1, йде спати
[T=3] Потік 4: shared_state['last_active'] = 4, йде спати
[T=4] Потік 0: прокидається, читає 'last_active' → бачить 4!
      Друкує: "Я потік 0, last_active = 4"
```

## Рішення: threading.local()

In [ ]:
import threading
import time

# threading.local() — виглядає глобально, але ІЗОЛЬОВАНЕ для кожного потоку
local_data = threading.local()

def safe_update_state(thread_id):
    # Записує у власну ізольовану копію local_data
    local_data.last_active = thread_id

    time.sleep(0.01)

    # Читає СВОЮ копію — завжди правильне значення!
    print(
        f"Я потік {thread_id}, "
        f"local last_active = {local_data.last_active}  "
        f"← ✅ завжди збіг!"
    )

threads = [threading.Thread(target=safe_update_state, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("\n✅ threading.local() гарантує ізоляцію!")

**Production use:** Flask та Django використовують `threading.local()` для per-request контексту (поточний user, database connection, request object).

# tqdm: Моніторинг Threading Pipeline в Реальному Часі

Починаючи з цієї секції ми додаємо `tqdm` до всіх прикладів.

## Що показує progress bar

```
Threads:  75%|███████████████▋     | 6/8 [00:02<00:01,  2.8it/s]  news=42  last=0.9s
           ▲           ▲              ▲      ▲       ▲              ▲
        відсоток    bar           крок  elapsed  швидкість     set_postfix()
```

## Три патерни у threading

```python
# 1. Звичайний цикл (sequential)
for url in tqdm(urls, desc='Scraping'):
    fetch(url)

# 2. as_completed ← найкорисніший у threading
for future in tqdm(as_completed(futures), total=len(futures), desc='Threads'):
    result = future.result()

# 3. executor.map
results = list(tqdm(executor.map(fn, items), total=len(items)))
```

**`tqdm.write()`** — лог поза progress bar, не ламає вигляд:  
```python
tqdm.write(f'✅ {url}: {count} результатів')
```


# Частина 9: ThreadPoolExecutor — Production Pattern

## Level 4: Архітектурне Мислення

Ручне управління потоками (`Thread()`, `.start()`, `.join()`) — не production pattern.

**Проблеми ручного управління:**
- Якщо один потік падає — виключення губляться
- Важко зібрати результати
- Немає обмеження на кількість потоків (можна спавнити тисячі!)
- Немає reuse потоків

**Production рішення:** `concurrent.futures.ThreadPoolExecutor`

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from tqdm import tqdm

def fetch_url(url):
    delay = random.uniform(0.1, 0.8)
    time.sleep(delay)
    if random.random() < 0.2:
        raise ConnectionError(f'Timeout для {url}')
    return {'url': url, 'status': 200, 'time': delay}

urls = [f'https://api.example.com/data/{i}' for i in range(10)]
results = []
errors  = []

with ThreadPoolExecutor(max_workers=4) as executor:
    future_to_url = {executor.submit(fetch_url, url): url for url in urls}

    # tqdm обгортає as_completed: бачимо кожен завершений Future
    progress = tqdm(
        as_completed(future_to_url),
        total=len(future_to_url),
        desc='ThreadPool',
        unit='req',
        colour='green',
    )
    for future in progress:
        url = future_to_url[future]
        try:
            r = future.result()
            results.append(r)
            tqdm.write(f'✅ {r["url"]}: {r["time"]:.2f}с')
            progress.set_postfix({'ok': len(results), 'err': len(errors)})
        except ConnectionError as e:
            errors.append(url)
            tqdm.write(f'❌ {url}: {e}')

print(f'\nУспішно: {len(results)}/{len(urls)}')
print(f'Помилки: {len(errors)}')


## Пояснення: Як працює ThreadPoolExecutor

Коли ми створюємо:

```python
with ThreadPoolExecutor(max_workers=4) as executor:
````

Python створює:

* пул потоків (thread pool)
* чергу задач
* механізм Future для отримання результатів

---

## Що відбувається всередині?

### 1. Ми submit-имо задачу

```python
future = executor.submit(fetch_data, url)
```

Це НЕ запускає новий thread кожного разу.

Замість цього:

* задача кладеться у внутрішню queue
* вільний worker thread бере її
* виконує
* повертає результат у Future

---

## Ментальна модель

```text
                submit(task)
                       │
                       ▼
        ┌─────────────────────────┐
        │      Work Queue         │
        │  [task1, task2, ...]    │
        └─────────────────────────┘
              ▲    ▲    ▲
              │    │    │
       ┌──────┘    │    └──────┐
       │           │           │

   ┌────────┐ ┌────────┐ ┌────────┐
   │Thread 1│ │Thread 2│ │Thread 3│
   └────────┘ └────────┘ └────────┘

       │           │           │
       ▼           ▼           ▼

   виконують задачі паралельно

       │
       ▼

┌────────────────────┐
│ Future object      │
│ .result()          │
│ .done()            │
│ .exception()       │
└────────────────────┘
```

---

## Що таке Future?

`Future` — це об’єкт, який представляє результат задачі у майбутньому.

```python
future = executor.submit(fetch_data, url)
```

Тут `future` — ще НЕ результат.

Це “обіцянка”, що результат з’явиться пізніше.

---

## Як отримати результат?

```python
result = future.result()
```

Якщо задача ще не завершилась:

* `.result()` заблокує потік
* і чекатиме завершення

---

## Чому ThreadPoolExecutor швидший?

Без pool:

```text
створити thread
запустити
знищити
створити thread
запустити
знищити
```

Це дорого.

ThreadPoolExecutor:

* створює потоки один раз
* перевикористовує їх
* worker threads живуть постійно

---

## Аналогія з рестораном

Уяви:

* Queue → список замовлень
* Threads → кухарі
* submit() → нове замовлення
* Future → номер замовлення

Кухарі постійно беруть нові задачі з черги.

Як тільки один кухар звільнився —
він бере наступну задачу.



## executor.map() — простіший синтаксис

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time
from tqdm import tqdm

def process_item(n):
    time.sleep(0.1)
    return n * n

items = list(range(20))
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as executor:
    # tqdm(executor.map(...), total=) — progress bar на map
    results = list(
        tqdm(
            executor.map(process_item, items),
            total=len(items),
            desc='executor.map',
            unit='task',
            colour='cyan',
        )
    )

elapsed = time.perf_counter() - start
print(f'Результати: {results}')
print(f'Час: {elapsed:.2f}с  (послідовно: {len(items)*0.1:.1f}с)')


# Частина 10: Producer/Consumer — Архітектура без спільного стану

## Найбезпечніший спосіб обміну даними між потоками

Замість того щоб ділити спільні змінні (і боротися з race conditions та locks) — передавай дані через **queue**.

`queue.Queue` — thread-safe черга. Внутрішньо вже використовує Lock + Condition. Тобі не потрібно управляти цим вручну.

Це архітектурний фундамент **Celery**, **RabbitMQ**, **Redis queues**.

In [ ]:
import threading
import queue
import time
import random
from tqdm import tqdm

work_queue = queue.Queue(maxsize=10)

# tqdm progress bar — оновлюється з будь-якого потоку через tqdm.write()
pbar = tqdm(total=8, desc='Producer/Consumer', unit='task', colour='magenta')

def producer(total_items):
    for i in range(total_items):
        item = {'id': i, 'data': f'task_{i}'}
        work_queue.put(item)
        tqdm.write(f'[Producer] → task_{i} (черга: {work_queue.qsize()})')
        time.sleep(0.05)
    work_queue.put(None)  # sentinel

def consumer(worker_id):
    while True:
        item = work_queue.get(block=True)
        if item is None:
            work_queue.put(None)
            break
        time.sleep(random.uniform(0.05, 0.15))
        tqdm.write(f'[Worker {worker_id}] ✅ task_{item["id"]} оброблено')
        pbar.update(1)         # ← оновлюємо bar з consumer потоку
        work_queue.task_done()

threads = [
    threading.Thread(target=producer,  args=(8,),   name='Producer'),
    threading.Thread(target=consumer,  args=(1,),   name='Worker-1'),
    threading.Thread(target=consumer,  args=(2,),   name='Worker-2'),
]
for t in threads: t.start()
for t in threads: t.join()

pbar.close()
print('\n✅ Всі задачі оброблено!')


## Пояснення: Чому Queue безпечніший за спільні змінні

| Підхід | Безпека | Складність | Production |
| ------ | ------- | ---------- | ---------- |
| Спільний dict/list без Lock | ❌ Race condition | Низька | Ніколи |
| Спільна змінна + Lock | ✅ Безпечно | Середня | Тільки прості випадки |
| `queue.Queue` | ✅✅ Безпечно | Низька | Рекомендовано |

**Ключова ідея:** Замість того щоб **ділити стан** — **передавай повідомлення**. Це фундаментальний принцип concurrent systems.

**Debugging tip:** Якщо `queue.task_done()` не викликається після кожного `get()` — `queue.join()` буде чекати вічно.


# Частина 11: Що відбувається під капотом

## 11.1 Context Switch на рівні OS

OS scheduler **не-детерміністичний** і може перервати потік **між будь-якими двома байткодовими інструкціями**.

**Як відбувається context switch:**

```
1. OS timer interrupt (hardware) — переривання кожні ~1-10мс
2. OS зберігає стан поточного потоку:
   - всі CPU регістри (включаючи Program Counter)
   - стек потоку
3. OS вибирає наступний потік (scheduling algorithm)
4. OS відновлює стан нового потоку
5. CPU продовжує виконання нового потоку
```

**Важливо:** OS не знає про Python. Він перериває на рівні **CPU інструкцій**, не Python рядків.

## 11.2 Як `threading.Lock()` працює всередині

```
lock.acquire():
    → Атомарна CAS (Compare-And-Swap) CPU інструкція
    → Якщо прапорець = 0: встановити 1, повернути True
    → Якщо прапорець = 1: 
       - syscall: futex(FUTEX_WAIT) 
       - OS переводить потік у Waiting state
       - CPU цей потік більше не отримує

lock.release():
    → Встановити прапорець = 0
    → syscall: futex(FUTEX_WAKE)
    → OS переводить один з waiting потоків у Runnable
```

**Висновок:** Lock — не Python конструкція. Це **OS механізм** (futex на Linux). Тому він ефективний: заблокований потік не витрачає CPU.

## 11.3 `dis` — дивимось на байткод

In [ ]:
import dis

# Дивимось скільки байткодових інструкцій у "простому" counter += 1
def simple_increment():
    counter = 0
    counter += 1

print("Байткод для counter += 1:")
print("=" * 40)
dis.dis(simple_increment)

print("\n" + "=" * 40)
print("Кожна рядок вище — окрема інструкція,")
print("між якими OS може зробити context switch.")

# Частина 12: Production Usage

## 12.1 Web Servers: Gunicorn Thread Workers

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time
import random
import threading
from tqdm import tqdm

request_context = threading.local()

def handle_request(request_id):
    request_context.request_id = request_id
    request_context.user_id    = f'user_{random.randint(1, 100)}'
    time.sleep(random.uniform(0.05, 0.2))
    return {
        'request_id': request_context.request_id,
        'user':       request_context.user_id,
        'status':     200,
    }

incoming_requests = list(range(1, 13))
start = time.perf_counter()

with ThreadPoolExecutor(max_workers=4, thread_name_prefix='gunicorn-worker') as pool:
    responses = list(
        tqdm(
            pool.map(handle_request, incoming_requests),
            total=len(incoming_requests),
            desc='Gunicorn workers',
            unit='req',
            colour='blue',
        )
    )

elapsed = time.perf_counter() - start
print(f'Оброблено {len(responses)} запитів за {elapsed:.2f}с')
print(f'(Послідовно: ~{len(responses)*0.125:.1f}с)')
for r in responses[:3]:
    print(f'  Request #{r["request_id"]}: {r["user"]} → {r["status"]}')


## 12.2 Правильний Debugging: logging замість print()

In [ ]:
import logging
import threading
import time
import random

# Налаштування logging з ім'ям потоку та timestamp
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s.%(msecs)03d [%(threadName)-12s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def worker_with_logging(task_id):
    """Воркер з правильним logging."""
    logger.debug(f"Починаю задачу {task_id}")

    duration = random.uniform(0.05, 0.15)
    time.sleep(duration)

    logger.info(f"Задача {task_id} завершена за {duration:.3f}с")
    return task_id

threads = [
    threading.Thread(
        target=worker_with_logging,
        args=(i,),
        name=f"Worker-{i}"  # Ім'я потоку у логах!
    )
    for i in range(4)
]

for t in threads:
    t.start()
for t in threads:
    t.join()

logger.info("Всі задачі завершені")

# Частина 13: Типові Помилки

## Помилка 1: Забути join() — Orphaned Threads

In [ ]:
import threading
import time

results = []

def slow_task(i):
    time.sleep(0.5)
    results.append(i)  # Пише після того як main може вже завершитись!

# ❌ ПОМИЛКА: забули join()
print("=== БЕЗ join() ===")
results.clear()
threads = [threading.Thread(target=slow_task, args=(i,)) for i in range(3)]
for t in threads:
    t.start()
# Main продовжує виконання — threads ще не завершились!
print(f"Результати (до join): {results}")

# Чекаємо щоб не зламати notebook
for t in threads:
    t.join()
print(f"Результати (після join): {results}")

# ✅ ПРАВИЛЬНО: join() одразу після start()
print("\n=== З join() ===")
results.clear()
threads = [threading.Thread(target=slow_task, args=(i,)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()  # Чекаємо
print(f"Результати: {results}")

## Помилка 2: Lock всередині циклу — Bottleneck

In [ ]:
import threading
import time

lock = threading.Lock()
result_list = []

def process_item(item):
    """Процесінг який займає час."""
    processed = item * item  # CPU робота
    return processed

# ❌ ПОГАНО: Lock тримається під час важкої роботи
def bad_worker(items):
    for item in items:
        with lock:  # Lock тримається ВЕСЬ час обробки!
            result = process_item(item)
            result_list.append(result)

# ✅ ДОБРЕ: Lock тільки для write операції
def good_worker(items):
    for item in items:
        result = process_item(item)  # Обробка поза Lock
        with lock:                   # Lock тільки для append
            result_list.append(result)

print("Правило: тримай critical section якомога коротшою.")
print("Lock тільки навколо МІНІМАЛЬНО необхідного коду.")

## Помилка 3: print() замість logging у multi-threaded коді

In [ ]:
import threading

# ❌ print() — потоки ділять sys.stdout
# Результат може бути: "HelWorldlo\n\n"
# print() НЕ є атомарним у багатопотоковому середовищі!

# ✅ logging — thread-safe, атомарний вивід з metadata
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="[%(threadName)s] %(message)s"
)

print("Правило: у multi-threaded коді завжди використовуй logging.")
print("logging модуль внутрішньо використовує Lock для thread-safety.")

# Частина 14: Практичні Вправи

## Вправа 1: Відтвори та виправ "Lost Update"

**Завдання:** Проведи banking simulation. Без Lock — спостережи race condition. Додай Lock у правильному місці.

**Підказка:** Lock потрібен навколо **всього** read-check-write циклу, а не тільки навколо write.

In [ ]:
import threading
import time

# === ВЕРСІЯ 1: БЕЗ ЗАХИСТУ (Race Condition) ===
bank_balance_unsafe = 100

def withdraw_unsafe(amount, name):
    global bank_balance_unsafe
    current = bank_balance_unsafe         # READ
    if current >= amount:
        time.sleep(0.05)                  # Симуляція мережевої затримки
        bank_balance_unsafe = current - amount  # WRITE
        print(f"[{name}] Знято {amount}£. Залишок: {bank_balance_unsafe}£")
    else:
        print(f"[{name}] Недостатньо коштів!")

print("=== БЕЗ LOCK (небезпечно) ===")
t1 = threading.Thread(target=withdraw_unsafe, args=(80, "User A"))
t2 = threading.Thread(target=withdraw_unsafe, args=(80, "User B"))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Фінальний баланс: {bank_balance_unsafe}£  (мало бути 20£, якщо тільки один успішний)")

# === ВЕРСІЯ 2: З LOCK (безпечно) ===
bank_balance_safe = 100
lock = threading.Lock()

def withdraw_safe(amount, name):
    global bank_balance_safe
    with lock:                            # Lock захищає ВЕСЬ цикл
        current = bank_balance_safe
        if current >= amount:
            time.sleep(0.05)
            bank_balance_safe = current - amount
            print(f"[{name}] Знято {amount}£. Залишок: {bank_balance_safe}£")
        else:
            print(f"[{name}] Недостатньо коштів!")

print("\n=== З LOCK (безпечно) ===")
t1 = threading.Thread(target=withdraw_safe, args=(80, "User A"))
t2 = threading.Thread(target=withdraw_safe, args=(80, "User B"))
t1.start(); t2.start()
t1.join(); t2.join()
print(f"Фінальний баланс: {bank_balance_safe}£")

## Вправа 2: Визнач та виправ Deadlock

**Завдання:** Нижче — код з deadlock. Знайди де саме і виправ через lock ordering.

**Підказка:** Знайди де потоки захоплюють locks у різному порядку.

In [ ]:
import threading
import time

account_lock = threading.Lock()
log_lock = threading.Lock()

# Ця функція виконує transfer і логування
# ЗНАЙДИ: де deadlock і чому

def transfer_with_log(from_acc, to_acc, amount):
    with account_lock:
        time.sleep(0.05)
        with log_lock:  # account_lock → log_lock
            print(f"Transfer {amount} from {from_acc} to {to_acc}")

def log_and_transfer(from_acc, to_acc, amount):
    with log_lock:
        time.sleep(0.05)
        with account_lock:  # log_lock → account_lock ← РІЗНИЙ порядок!
            print(f"Logged: {amount} from {from_acc} to {to_acc}")

t1 = threading.Thread(target=transfer_with_log, args=("A", "B", 100))
t2 = threading.Thread(target=log_and_transfer, args=("C", "D", 200))

t1.start(); t2.start()

t1.join(timeout=2)
t2.join(timeout=2)

if t1.is_alive() or t2.is_alive():
    print("💀 Deadlock! Знайдено проблему.")
    print("Рішення: обидві функції мають захоплювати locks у порядку: account_lock → log_lock")

# --- Виправлена версія ---
print("\n=== ВИПРАВЛЕНА ВЕРСІЯ ===")

def safe_transfer(from_acc, to_acc, amount, label):
    # Завжди account_lock спочатку, потім log_lock
    with account_lock:
        time.sleep(0.05)
        with log_lock:
            print(f"[{label}] {amount} from {from_acc} to {to_acc}")

t1 = threading.Thread(target=safe_transfer, args=("A", "B", 100, "transfer"))
t2 = threading.Thread(target=safe_transfer, args=("C", "D", 200, "log"))

t1.start(); t2.start()
t1.join(); t2.join()
print("✅ Deadlock виправлено!")

## Вправа 3: Concurrent API Scraper

**Завдання:** Побудуй concurrent scraper, що завантажує дані з кількох ендпоінтів паралельно.

**Вимоги:**
1. Використати `ThreadPoolExecutor` з max_workers=5
2. Обробляти помилки (timeout, connection error)
3. Зібрати результати та помилки окремо
4. Замір часу виконання

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import random
from tqdm import tqdm

ENDPOINTS = [
    {'url': 'https://api.example.com/users',     'latency': 0.3},
    {'url': 'https://api.example.com/products',  'latency': 0.5},
    {'url': 'https://api.example.com/orders',    'latency': 0.2},
    {'url': 'https://api.example.com/inventory', 'latency': 0.8},
    {'url': 'https://api.slow.com/data',          'latency': 1.5},
    {'url': 'https://api.broken.com/endpoint',   'latency': 0.1, 'fail': True},
    {'url': 'https://api.example.com/reports',   'latency': 0.4},
]

def fetch_endpoint(endpoint):
    time.sleep(endpoint['latency'])
    if endpoint.get('fail'):
        raise ConnectionError(f'Connection refused: {endpoint["url"]}')
    return {'url': endpoint['url'],
            'data': f'[{random.randint(100,999)} records]',
            'latency_ms': endpoint['latency'] * 1000}

start     = time.perf_counter()
successes = []
failures  = []

with ThreadPoolExecutor(max_workers=5) as executor:
    future_map = {executor.submit(fetch_endpoint, ep): ep for ep in ENDPOINTS}

    for future in tqdm(
        as_completed(future_map),
        total=len(future_map),
        desc='Scraping endpoints',
        unit='url',
        colour='green',
    ):
        ep = future_map[future]
        try:
            r = future.result()
            successes.append(r)
            tqdm.write(f'✅ {ep["url"]}: {r["data"]} ({r["latency_ms"]:.0f}мс)')
        except ConnectionError as e:
            failures.append({'url': ep['url'], 'error': str(e)})
            tqdm.write(f'❌ {ep["url"]}: {e}')

elapsed = time.perf_counter() - start
seq_time = sum(ep['latency'] for ep in ENDPOINTS)
print(f'\nЕндпоінти: {len(ENDPOINTS)}')
print(f'Успішно:   {len(successes)}/{len(ENDPOINTS)}')
print(f'Час:       {elapsed:.2f}с  vs  послідовно {seq_time:.2f}с')
print(f'Прискорення: {seq_time/elapsed:.1f}x')


# Частина 15: Поглиблені Вправи — Системне Мислення

П'ять вправ побудованих за принципом **Predict → Observe → Explain**.
Кожна вправа розроблена щоб зламати одне хибне уявлення про threading.

| Вправа | Концепція | Misconception |
| ------ | --------- | ------------- |
| 4 | Race Condition (100 threads) | `+=` атомарна |
| 5 | Deadlock (Bank Transfer) | Locks завжди безпечні |
| 6 | executor.map ordering | map повертає results по мірі готовності |
| 7 | Swallowed Exceptions | Виключення з потоку одразу видно |
| 8 | Zombie Workers / Queue | q.join() вбиває потоки |

## Вправа 4: Ілюзія Атомарності — Analytics Dashboard

### Контекст

Ти будуєш analytics dashboard. Кожен клік користувача запускає background thread,
що інкрементує глобальний лічильник. Симулюємо high-traffic event:
100 потоків, кожен робить 100,000 інкрементів.

### Prediction Quiz — передбач ПЕРЕД запуском

Що виведе `print(f"Total clicks: {counter}")`?

- **A)** Рівно 10,000,000
- **B)** Рівно 100,000
- **C)** Випадкове число, трохи менше 10,000,000
- **D)** `RuntimeError: ConcurrentModification`

*Запустиш клітинку — побачиш відповідь.*

In [ ]:
import threading

counter = 0  # Живе на Heap — спільна пам'ять для всіх 100 потоків

def add_clicks():
    global counter
    for _ in range(100_000):
        counter += 1  # Виглядає як 1 крок → насправді READ + ADD + WRITE

threads = [threading.Thread(target=add_clicks) for _ in range(100)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Очікували: 10,000,000")
print(f"Отримали:  {counter:,}")
print(f"Втрачено:  {10_000_000 - counter:,} кліків ({(10_000_000 - counter)/10_000_000*100:.1f}%)")

### Відповідь: **C — Випадкове число менше 10,000,000**

### Як виправити — з виміром вартості Lock

Виправлення: `threading.Lock()` навколо critical section.
Але Lock коштує: паралельна програма стає послідовною на critical section.
Виміряємо реальну вартість:

In [ ]:
import threading
import time

# ── БЕЗ Lock (race condition) ──
counter_unsafe = 0

def add_unsafe():
    global counter_unsafe
    for _ in range(100_000):
        counter_unsafe += 1

start = time.perf_counter()
threads = [threading.Thread(target=add_unsafe) for _ in range(100)]
for t in threads: t.start()
for t in threads: t.join()
time_unsafe = time.perf_counter() - start

# ── З Lock (thread-safe) ──
counter_safe = 0
lock = threading.Lock()

def add_safe():
    global counter_safe
    for _ in range(100_000):
        with lock:        # Критична секція: тільки один потік тут
            counter_safe += 1

start = time.perf_counter()
threads = [threading.Thread(target=add_safe) for _ in range(100)]
for t in threads: t.start()
for t in threads: t.join()
time_safe = time.perf_counter() - start

print(f"БЕЗ Lock:  результат={counter_unsafe:>12,}  час={time_unsafe:.2f}с")
print(f"З Lock:    результат={counter_safe:>12,}  час={time_safe:.2f}с")
print(f"\nLock вартість: {time_safe/time_unsafe:.1f}x повільніше")
print("(Це ціна coректності. У production — мінімізуй critical section)")

### Рефлексія

**Чому Lock уповільнює програму настільки сильно?**

З Lock — всі 100 потоків стоять у черзі на кожен `+=`. Critical section займає
мікросекунди, але при 10,000,000 операцій — накопичується.

**Production рішення:** Замість глобального лічильника з Lock використовують
атомарні операції (`multiprocessing.Value`) або lock-free структури (Redis `INCR`).

**Real-world:** В e-commerce inventory системах non-atomic update дозволяє
двом покупцям купити останній товар одночасно — класичний overselling bug.

## Вправа 5: Смертельне Обіймання — Bank Transfer

### Контекст

Ти пишеш скрипт для переказу коштів між двома рахунками.
Для thread-safety: спочатку лочиш рахунок-джерело, потім рахунок-призначення.
Alice переказує Bob, а Bob одночасно переказує Alice.

### Prediction Quiz

Що станеться?

- **A)** Обидва перекази виконаються послідовно
- **B)** `RuntimeError: Maximum lock depth exceeded`
- **C)** Програма зависне назавжди
- **D)** Один переказ виконається, другий відкинеться випадково

In [ ]:
import threading
import time

class Account:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance
        self.lock = threading.Lock()

def transfer(acc_from, acc_to, amount):
    with acc_from.lock:              # Захоплюємо перший lock
        time.sleep(0.1)              # Симуляція DB затримки (тут OS робить switch)
        with acc_to.lock:            # Захоплюємо другий lock — тут deadlock!
            acc_from.balance -= amount
            acc_to.balance += amount
            print(f"Переказано ${amount}: {acc_from.name} → {acc_to.name}")

alice = Account("Alice", 100)
bob   = Account("Bob",   100)

t1 = threading.Thread(target=transfer, args=(alice, bob, 10))
t2 = threading.Thread(target=transfer, args=(bob, alice, 20))

t1.start(); t2.start()

# Timeout щоб не зависнути у notebook
t1.join(timeout=2)
t2.join(timeout=2)

if t1.is_alive() or t2.is_alive():
    print("💀 DEADLOCK! Потоки живі після timeout:")
    print(f"   Alice→Bob: {t1.is_alive()}  |  Bob→Alice: {t2.is_alive()}")
    print()
    print("Lock ownership:")
    print("  [Thread 1] → (HOLDS) → [Alice.lock]")
    print("      |                        ^")
    print("   (WAITS FOR)            (WAITS FOR)")
    print("      V                        |")
    print("  [Bob.lock]  ← (HOLDS) ← [Thread 2]")

### Відповідь: **C — Програма зависне назавжди**

Deadlock виник через **Circular Wait**: обидва потоки тримають по одному lock
і чекають lock, який тримає інший.

### Виправлення: Lock Ordering через id()

Обидва потоки завжди захоплюють locks у **одному глобальному порядку**
(наприклад, за `id()` об'єкта — детермінований, не залежить від назви).

In [ ]:
import threading
import time

class Account:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance
        self.lock = threading.Lock()

def safe_transfer(acc_from, acc_to, amount):
    # Завжди захоплюємо locks у порядку id() — незалежно від напряму переказу
    first, second = sorted([acc_from, acc_to], key=lambda a: id(a.lock))
    with first.lock:
        time.sleep(0.1)
        with second.lock:
            acc_from.balance -= amount
            acc_to.balance += amount
            print(f"✅ Переказано ${amount}: {acc_from.name} → {acc_to.name}  "
                  f"(Аліса: ${alice.balance}, Боб: ${bob.balance})")

alice = Account("Alice", 100)
bob   = Account("Bob",   100)

t1 = threading.Thread(target=safe_transfer, args=(alice, bob, 10))
t2 = threading.Thread(target=safe_transfer, args=(bob, alice, 20))

t1.start(); t2.start()
t1.join(); t2.join()

print("\n✅ Deadlock виправлено через lock ordering!")

## Вправа 6: Секретно Послідовний Пул — executor.map Ordering

### Контекст

Скрапімо 4 сайти через `ThreadPoolExecutor`. Сайт 0 недоступний — таймаут 10с.
Сайти 1-3 швидкі — 1с. Використовуємо `executor.map`.

### Prediction Quiz

Коли з'явиться результат **швидкого Сайту 1** на екрані?

- **A)** Через 1с — одразу як він завершиться
- **B)** Через 10с — після того як Сайт 0 поверне результат
- **C)** Скрипт впаде через 10с з timeout exception

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def scrape(site_id):
    delay = 10 if site_id == 0 else 1  # Сайт 0 "не відповідає" 10 секунд
    time.sleep(delay)
    return f"Дані Сайту {site_id} (затримка: {delay}с)"

print("Запускаю 4 потоки одночасно...")
t_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as executor:
    # executor.map зберігає ПОРЯДОК ВХІДНИХ ДАНИХ — не порядок завершення!
    results = executor.map(scrape, range(4))

    for res in results:
        elapsed = time.perf_counter() - t_start
        print(f"[{elapsed:.1f}с] Отримано: {res}")

### Відповідь: **B — Результат Сайту 1 з'явиться через 10с**

**Misconception:** `executor.map` повертає результати по мірі готовності.

**Реальність:** `map` суворо зберігає **порядок вхідного ітерабл**.
Ітерація блокується на першому незавершеному future.

```
[T=0]    4 потоки стартували одночасно
[T=1]    Сайти 1, 2, 3 завершились — результати в пам'яті, але чекають
[T=1+]   for-loop запитує результат Сайту 0 → БЛОКУЄТЬСЯ (він ще не готовий)
[T=10]   Сайт 0 завершився → loop виводить 0, потім миттєво 1, 2, 3
```

Все це — щоб Сайти 1-3 чекали 9с зайво, займаючи пам'ять.

**Рішення для виведення по мірі готовності — `as_completed`:**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from tqdm import tqdm

def scrape(site_id):
    delay = 10 if site_id == 0 else 1
    time.sleep(delay)
    return f'Дані Сайту {site_id}'

print('as_completed + tqdm: результати одразу при готовності')
t_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(scrape, i): i for i in range(4)}

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc='Scraping',
        unit='site',
        colour='cyan',
    ):
        site_id = futures[future]
        elapsed = time.perf_counter() - t_start
        tqdm.write(f'[{elapsed:.1f}с] Сайт {site_id}: {future.result()}')

print(f'\nЗагальний час: {time.perf_counter()-t_start:.1f}с')
print('(Сайти 1-3 з\'явились через 1с, Сайт 0 — через 10с)')


## Вправа 7: Привид у Потоці — Проковтнуті Виключення

### Контекст

Використовуєш `ThreadPoolExecutor` для фонових математичних обчислень.
Дані брудні: десь є ділення на нуль.

### Prediction Quiz

Що побачиш у Jupyter Notebook?

- **A)** Програма негайно впаде з `ZeroDivisionError` traceback
- **B)** Потік тихо помре, нічого не виведеться, програма завершиться нібито успішно
- **C)** Потік автоматично перезапуститься

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def dangerous_task(divisor):
    print(f"[Task {divisor}] Стартував...")
    time.sleep(0.5)
    result = 100 / divisor  # ZeroDivisionError якщо divisor=0!
    print(f"[Task {divisor}] Завершив: 100/{divisor} = {result}")
    return result

with ThreadPoolExecutor(max_workers=3) as executor:
    future_a = executor.submit(dangerous_task, 5)   # OK
    future_b = executor.submit(dangerous_task, 0)   # ZeroDivisionError!
    future_c = executor.submit(dangerous_task, 2)   # OK

    # Не викликаємо future_b.result() — виключення буде ПРОКОВТНУТО
    time.sleep(2)

print("\nГоловна програма завершилась успішно.")
print("(Але Task 0 впав! Де його помилка?)")

### Відповідь: **B — Потік тихо помре без жодного повідомлення**

**Misconception:** Виключення з потоку автоматично з'являються в main thread.

**Реальність:** `executor.submit()` перехоплює виключення, загортає у `Future`
і **проковтує** якщо ніхто не викликає `future.result()`.

Це найнебезпечніший клас багів у distributed systems — **silent failures**.

### Правильна обробка: завжди перевіряй future.result()

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def dangerous_task(divisor):
    time.sleep(0.3)
    return 100 / divisor  # Впаде якщо divisor=0

divisors = [5, 0, 2, 0, 10]  # Два нулі — дві тихі помилки

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(dangerous_task, d): d for d in divisors}

    for future in futures:
        divisor = futures[future]
        try:
            result = future.result()  # future.result() повторно кидає виключення!
            print(f"✅ 100/{divisor} = {result}")
        except ZeroDivisionError:
            print(f"❌ 100/{divisor} → ZeroDivisionError (зловлено в main thread)")

print("\nReal-world: email/notification сервіси можуть тихо дропати задачі")
print("тижнями без logs — поки юзери не почнуть скаржитись.")

## Вправа 8: Зомбі Воркер — Queue Synchronization

### Контекст

Producer/Consumer pipeline: producer генерує 2 задачі, 3 worker threads їх споживають.
Чекаємо завершення через `q.join()`.

### Prediction Quiz

Чи завершиться скрипт і поверне курсор?

- **A)** Так — після 2 задач потоки виходять і програма завершується
- **B)** Ні — програма зависне назавжди
- **C)** Так — але кине `queue.Empty`

In [ ]:
import threading
import queue
import time

q = queue.Queue()

def worker(worker_id):
    while True:
        task = q.get(block=True)  # Блокується якщо черга порожня
        print(f"[Worker {worker_id}] Обробляє: {task}")
        time.sleep(0.3)
        q.task_done()
        # Потік повертається в while True → знову чекає у q.get()
        # Нема виходу з циклу → Зомбі!

# Запускаємо 3 потоки
threads = []
for i in range(3):
    t = threading.Thread(target=worker, args=(i,))
    # t.daemon = True  ← що станеться якщо розкоментувати?
    threads.append(t)
    t.start()

# Тільки 2 задачі для 3 воркерів
q.put("Task A")
q.put("Task B")

print("Чекаю завершення черги...")
q.join()  # Розблокується коли всі task_done() викликані
print("Черга порожня! Але потоки ще живі — зомбі...")
print(f"Живих потоків: {sum(1 for t in threads if t.is_alive())}")
print("\n(Jupyter не зависне бо non-daemon threads не блокують IPython kernel)")
print("В звичайному .py скрипті — програма ніколи б не завершилась!")

### Відповідь: **B — В реальному скрипті програма зависне**

**Misconception:** `q.join()` завершує background threads.

**Реальність:** `q.join()` тільки розблоковує **main thread** коли всі
`task_done()` викликані. Worker threads залишаються живими, застрягши у `while True: q.get()`.

Python не дозволяє скрипту завершитись поки живі **non-daemon** threads.

### Два рішення:

In [ ]:
import threading
import queue
import time
from tqdm import tqdm

# ── РІШЕННЯ 1: Poison Pill (Sentinel) ──
print('=== Рішення 1: Poison Pill ===')

q1   = queue.Queue()
STOP = object()
pbar = tqdm(total=8, desc='Poison Pill', unit='task', colour='magenta')

def worker_with_stop(worker_id):
    while True:
        task = q1.get()
        if task is STOP:
            q1.put(STOP)
            q1.task_done()
            tqdm.write(f'[Worker {worker_id}] Отримав STOP ✋')
            break
        tqdm.write(f'[Worker {worker_id}] ✅ {task}')
        time.sleep(0.1)
        pbar.update(1)
        q1.task_done()

threads = [threading.Thread(target=worker_with_stop, args=(i,)) for i in range(3)]
for t in threads: t.start()

for i in range(8):
    q1.put(f'Task {i}')
q1.put(STOP)

q1.join()
for t in threads: t.join()
pbar.close()
print(f'Живих потоків: {sum(1 for t in threads if t.is_alive())} ✅\n')

# ── РІШЕННЯ 2: Daemon Threads ──
print('=== Рішення 2: Daemon Threads ===')

q2   = queue.Queue()
pbar2 = tqdm(total=2, desc='Daemon tasks', unit='task', colour='blue')

def simple_worker(worker_id):
    while True:
        task = q2.get(block=True)
        tqdm.write(f'[Daemon {worker_id}] {task}')
        time.sleep(0.1)
        pbar2.update(1)
        q2.task_done()

for i in range(3):
    threading.Thread(target=simple_worker, args=(i,), daemon=True).start()

q2.put('Task X')
q2.put('Task Y')
q2.join()
pbar2.close()
print('✅ Daemon threads автоматично вбито OS.')


# Частина 16: Архітектурні Сценарії — Advanced Deadlocks

Два сценарії для переходу від **написання коду** до **архітектурного мислення**.
Deadlocks — не "помилки друку". Це фундаментальні дефекти в тому,
як компоненти системи ділять та конкурують за ресурси.

| Сценарій | Тип Deadlock | Складність |
| -------- | ------------ | ---------- |
| 1 | Circular Wait (Inconsistent Lock Ordering) | ⭐⭐⭐ |
| 2 | Nested Locks + Hidden Blocking (Bounded Queue) | ⭐⭐⭐⭐ |

## Сценарій 1: Bank Transfer — Circular Wait

### Проблема

Deadlock через **непослідовне захоплення locks**.
Найкласичніша threading пастка в фінансових системах.

### Prediction Quiz

Alice переказує Bob $10 **одночасно** як Bob переказує Alice $20.
Що трапиться?

> Підказка: подивись уважно на порядок lock acquisition у кожному потоці.

In [ ]:
import threading
import time

class Account:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance
        self.lock = threading.Lock()

def transfer(acc_from, acc_to, amount):
    with acc_from.lock:          # Thread 1: захоплює alice.lock
        time.sleep(0.1)          # OS context switch! Thread 2 запускається
        with acc_to.lock:        # Thread 1 хоче bob.lock — але Thread 2 його тримає!
            acc_from.balance -= amount
            acc_to.balance   += amount
            print(f"Переказано ${amount}: {acc_from.name} → {acc_to.name}")

alice = Account("Alice", 500)
bob   = Account("Bob",   500)

t1 = threading.Thread(target=transfer, args=(alice, bob,   10))
t2 = threading.Thread(target=transfer, args=(bob,   alice, 20))

t1.start(); t2.start()
t1.join(timeout=2); t2.join(timeout=2)

if t1.is_alive() or t2.is_alive():
    print("💀 DEADLOCK — жоден переказ не виконався")
    print()
    print("  Lock ownership visualization:")
    print("  [Thread 1] ──(HOLDS)──► [Alice.lock]")
    print("      │                         ▲")
    print("  (WAITS FOR)             (WAITS FOR)")
    print("      ▼                         │")
    print("  [Bob.lock]  ◄──(HOLDS)── [Thread 2]")
    print()
    print("  Кожна стрілка утворює замкнутий цикл → Circular Wait")

### Аналіз: Чому це Heisenbug

Видали `time.sleep(0.1)` — код може працювати **9,999 разів з 10,000**.
Deadlock залежить від мікросекундного timing OS scheduler.
Локально не відтворюється, під навантаженням на prod — зависає.

### Архітектурне рішення: Global Lock Ordering

**Математичне правило:** Ніколи не може бути Circular Wait якщо всі потоки
захоплюють ресурси в одному глобальному порядку.

Порядок визначається через `id()` — унікальний ідентифікатор об'єкта в пам'яті:

In [ ]:
import threading
import time

class Account:
    def __init__(self, name, balance):
        self.name = name
        self.balance = balance
        self.lock = threading.Lock()

def safe_transfer(acc_from, acc_to, amount):
    # Глобальний порядок: завжди lock з меншим id() першим
    # Незалежно від напряму переказу — порядок ДЕТЕРМІНОВАНИЙ
    first, second = sorted([acc_from, acc_to], key=lambda a: id(a.lock))

    with first.lock:
        time.sleep(0.1)  # OS context switch — але deadlock неможливий!
        with second.lock:
            acc_from.balance -= amount
            acc_to.balance   += amount
            print(f"✅ ${amount}: {acc_from.name}(${acc_from.balance}) "
                  f"→ {acc_to.name}(${acc_to.balance})")

alice = Account("Alice", 500)
bob   = Account("Bob",   500)

transfers = [
    threading.Thread(target=safe_transfer, args=(alice, bob,   10)),
    threading.Thread(target=safe_transfer, args=(bob,   alice, 20)),
    threading.Thread(target=safe_transfer, args=(alice, bob,   30)),
]

for t in transfers: t.start()
for t in transfers: t.join()

print(f"\nАліса: ${alice.balance}  Боб: ${bob.balance}")
print("Сума незмінна:", alice.balance + bob.balance, "== 1000")
print("\n✅ Lock ordering виключає Circular Wait математично!")

## Сценарій 2: Bounded Queue — Прихований Deadlock через Вкладені Locks

### Проблема

Deadlock без **явного** circular wait — через `queue.put()` всередині Lock.
Найкращий приклад **прихованого** blocking механізму.

### Prediction Quiz

Consumer активно споживає задачі. Producer кладе в чергу.
Чи producer успішно покладе всі 5 задач у чергу розміром 2?

> Підказка: подивись **де знаходиться Lock** відносно `queue.put()`.

In [ ]:
import threading
import queue
import time

data_queue  = queue.Queue(maxsize=2)  # Черга тільки на 2 елементи!
state_lock  = threading.Lock()

def producer():
    for i in range(5):
        with state_lock:              # Захоплюємо lock...
            print(f"[Producer] Кладу {i}...")
            data_queue.put(i, block=True)  # ...і БЛОКУЄМОСЬ якщо черга повна!
            print(f"[Producer] Поклав {i}")  # До 3-го елемента не дійдемо

def consumer():
    while True:
        with state_lock:              # Хоче lock — але producer його тримає!
            item = data_queue.get(block=True)
            print(f"[Consumer] Спожив {item}")
            data_queue.task_done()

threading.Thread(target=producer).start()
threading.Thread(target=consumer, daemon=True).start()

# Чекаємо трохи і перевіряємо стан
time.sleep(1)
print(f"\nРозмір черги: {data_queue.qsize()} (заповнена — producer застряг)")
print("💀 DEADLOCK через Lock всередині blocking queue.put()")

### Аналіз: Чому це важко побачити

```
[Producer]  ──(HOLDS)──► [state_lock]
     │                         ▲
 (WAITS FOR)             (WAITS FOR)
     ▼                         │
[Queue Space] ◄──(CONTROLS)── [Consumer]
```

Новачок подивиться і скаже: *"Тут тільки один Lock — circular wait неможливий!"*

**Помилка:** `queue.put(block=True)` — це **неявний lock**. Consumer "тримає"
простір у черзі (тільки він може звільнити місце), але чекає `state_lock`.
Producer тримає `state_lock` і чекає простір у черзі.

**Золоте правило:**
> Ніколи не виконуй IO, network запити, або blocking `queue.put()`
> **всередині** `with lock:`. Lock має оточувати тільки мікросекунди модифікації стану.

### Виправлення: Lock тільки для мутації стану, не для IO

In [ ]:
import threading
import queue
import time

data_queue = queue.Queue(maxsize=2)
state_lock = threading.Lock()
processed_count = 0  # Спільний стан, що дійсно потребує захисту

def producer_fixed():
    for i in range(5):
        # queue.put() ПОЗА lock — blocking IO не тримає lock!
        data_queue.put(i, block=True)
        print(f"[Producer] Поклав {i} (черга: {data_queue.qsize()})")

def consumer_fixed():
    global processed_count
    while True:
        item = data_queue.get(block=True)  # IO поза lock
        time.sleep(0.1)                    # Обробка поза lock

        with state_lock:                   # Lock тільки для запису лічильника
            processed_count += 1

        print(f"[Consumer] Спожив {item} (всього: {processed_count})")
        data_queue.task_done()

prod = threading.Thread(target=producer_fixed)
cons = threading.Thread(target=consumer_fixed, daemon=True)

prod.start(); cons.start()
prod.join()
data_queue.join()  # Чекаємо поки всі task_done() викликані

print(f"\n✅ Всі задачі оброблено. processed_count = {processed_count}")
print("Lock тільки навколо мутації стану — жодного прихованого blocking!")

# Частина 17: Рефлексійні Питання

## Питання для глибокого розуміння

Відповідай без коду — тільки своїми словами.

### Питання 1

> Якщо GIL не дозволяє двом потокам виконувати Python байткод одночасно — чому ми все одно отримуємо race conditions?

**Підказка:** Подумай про те, що відбувається між читанням і записом змінної. GIL передається між байткодами, не між Python рядками.

---

### Питання 2

> `time.sleep()` відпускає GIL. Але якщо потік тримає `lock` і входить у `time.sleep()` — чи відпускає він lock?

**Підказка:** GIL і Lock — різні механізми. Що трапиться з потоками, що чекають на цей lock?

---

### Питання 3

> Чому безпечніше передавати дані між потоками через `queue.Queue`, а не через спільний глобальний словник?

**Підказка:** Queue внутрішньо вже містить синхронізацію. Що потрібно робити вручну при спільному словнику?

---

*(Відповіді у `lesson_documentation.md`, розділ 15)*

# Summary

## Що ти вивчив у цьому уроці

### Концепції

| Концепція | Суть |
| --- | --- |
| **Thread vs Process** | Потоки ділять Heap; кожен має власний Stack |
| **GIL** | Один потік виконує Python байткод. IO відпускає GIL |
| **Race Condition** | +=` — не атомарна. OS перериває між Read-Add-Write |
| **Lock** | Critical section: тільки один потік всередині |
| **Deadlock** | Circular wait — threads сплять вічно. Рішення: lock ordering |
| **Thread-Local** | threading.local() — глобальна назва, ізольоване значення |


## Коли що використовувати

```
IO-bound (мережа, файли, БД)
  → threading або asyncio
  → ThreadPoolExecutor — production pattern

CPU-bound (математика, ML, encoding)
  → multiprocessing
  → ProcessPoolExecutor

Обмін даними між потоками
  → queue.Queue (найбезпечніше)
  → threading.Lock (тільки якщо Queue не підходить)
```

### Правила Production

1. **Завжди `.join()`** — не залишай orphaned threads
2. **`logging` замість `print()`** — в multi-threaded коді
3. **Lock ordering** — однаковий порядок у всіх потоках
4. **Мінімальна critical section** — lock тільки навколо необхідного
5. **`ThreadPoolExecutor`** замість ручного управління потоками
6. **`queue.Queue`** для обміну даними без race conditions

---


## Повна структура уроку

## Повна структура уроку

| Частина | Тема | Зміст |
| --- | --- | --- |
| 1-4 | Теорія · GIL · IO-bound · Недетермінізм | Концепції |
| 5-10 | Race Condition → Lock → Deadlock → TLS → ThreadPool → Queue | Код + Timeline |
| 11-14 | Капот, Production, Помилки, Базові вправи | dis · logging · gunicorn |
| 15 | Поглиблені вправи | Atomicity · Deadlock · map · Exceptions · Zombies |
| 16 | Архітектурні сценарії | Bank Transfer · Bounded Queue |
---

## Наступний урок

**Урок 33: multiprocessing**

Ми побачили, що threading добре для IO-bound задач — але у нього є обмеження:
- Кожен потік використовує ~8MB пам'яті
- Context switching дорогий
- При 10,000 concurrent connections — 10,000 потоків = проблема

`asyncio` вирішує цю проблему інакше — через **coroutines та event loop**.
Один потік, мільйони concurrent операцій.